# ⚓ Módulo 2: Proceso de Análisis de Datos (AIS)
### Maestría en Logística y Transporte Marítimo — UMIP
#### Preparado por: Jcenteno

La limpieza de datos de señales AIS (Automatic Identification System) es fundamental. Una mala limpieza puede llevar a conclusiones erróneas sobre el tráfico en el Canal.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
print("✅ Entorno listo")

## 1. Carga e Identificación de Errores
Cargamos datos con errores deliberados (Nulos, Duplicados, Outliers).

In [ ]:
def cargar_ais():
    try:
        # Intentar cargar desde GitHub (Colab) o Local
        url = "https://jacenteno.github.io/Analisis_Datos_Maritimo/02_Datasets/ais_muestra_canal.csv"
        try:
            df = pd.read_csv(url)
            print("✅ Datos cargados desde GitHub")
        except:
            df = pd.read_csv("../02_Datasets/ais_muestra_canal.csv")
            print("🏠 Datos cargados localmente")
    except:
        # Generación sintética si falla
        df = pd.DataFrame({
            'mmsi': [355789]*500 + [371234]*500,
            'lat': np.random.uniform(7.0, 9.5, 1000),
            'lon': np.random.uniform(-80.5, -77.5, 1000),
            'velocidad_kn': np.random.normal(12, 3, 1000)
        })
        # Inyectar errores
        df.loc[10:20, 'velocidad_kn'] = np.nan
        df.loc[50:60, 'lat'] = 25.0  # Outlier en el Ártico
    return df

df_ais = cargar_ais()
print(f"Registros iniciales: {len(df_ais)}")
print(df_ais.isnull().sum())

## 2. Proceso de Limpieza (Geofencing)
Solo nos interesan los buques dentro de la caja de coordenadas de Panamá.

In [ ]:
# 1. Eliminar duplicados exactos
df_ais = df_ais.drop_duplicates()

# 2. Geofencing (Caja de Panamá)
lat_range = (7.0, 9.5)
lon_range = (-80.5, -77.5)

df_clean = df_ais[
    (df_ais['lat'].between(*lat_range)) &
    (df_ais['lon'].between(*lon_range))
].copy()

# 3. Imputación de velocidad (Usamos la mediana por buque)
df_clean['velocidad_kn'] = df_clean.groupby('mmsi')['velocidad_kn'].transform(lambda x: x.fillna(x.median()))

print(f"Registros tras limpieza: {len(df_clean)}")

## 3. Visualización de Calidad
Comparamos la distribución de velocidades antes y después.

In [ ]:
plt.figure(figsize=(10, 5))
sns.kdeplot(df_ais['velocidad_kn'], label="Original", shade=True)
sns.kdeplot(df_clean['velocidad_kn'], label="Limpio", shade=True)
plt.title("Impacto de la limpieza en la distribución de velocidad")
plt.legend()
plt.show()

## 🚀 Laboratorio M2
**Reto**: Identifica qué buque (MMSI) tiene la mayor cantidad de saltos de posición imposibles (velocidad calculada > 40 nudos entre puntos) y elimina esos picos del análisis.